# Colab용 Chest X-ray 정상/비정상 이진분류 CNN

이 노트북은 Google Colab에서 바로 실행하기 위한 간단 버전입니다.

실행 전 확인할 것:

1. Colab 메뉴에서 `런타임 > 런타임 유형 변경 > GPU` 선택
2. Google Drive에 `Chest X_Ray Dataset.zip` 업로드
3. 아래 셀들을 위에서 아래로 순서대로 실행

## 1. 라이브러리 불러오기와 GPU 확인

In [ ]:
import copy
import os
import random
import shutil
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Google Drive 연결과 압축 해제

`ZIP_PATH`는 Drive에 올린 zip 파일 경로입니다. 지금은 `내 드라이브` 바로 아래에 있는 경우로 맞춰두었습니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# 파일 위치가 다르면 이 줄만 수정하세요.
ZIP_PATH = "/content/drive/MyDrive/Chest X_Ray Dataset.zip"
EXTRACT_DIR = Path("/content/xray")

# 이전에 잘못 풀린 폴더가 남아 있을 수 있으므로 매번 깨끗하게 다시 풉니다.
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("압축 해제 완료:", EXTRACT_DIR)

## 3. 데이터 폴더 자동 찾기와 하이퍼파라미터 설정

하이퍼파라미터는 모델 학습 전에 사람이 정해주는 설정값입니다. 여기서는 Colab T4 GPU 기준으로 무난하게 학습되는 값을 사용했습니다.

- `IMAGE_SIZE=224`: X-ray 이미지를 224x224 크기로 맞춥니다. 너무 작으면 병변 정보가 줄고, 너무 크면 학습이 느려집니다.
- `BATCH_SIZE=32`: 한 번에 GPU에 넣는 이미지 수입니다. GPU 메모리가 여유 있으면 64로 올려도 됩니다.
- `EPOCHS=20`: 전체 학습 데이터를 최대 20번 반복해서 봅니다. 중간에 성능이 좋아지지 않으면 early stopping으로 멈춥니다.
- `LEARNING_RATE=1e-4`: 가중치를 업데이트하는 속도입니다. 너무 크면 학습이 불안정하고, 너무 작으면 오래 걸립니다.
- `WEIGHT_DECAY=1e-4`: 과적합을 줄이기 위한 규제값입니다.
- `DROPOUT=0.3`: 마지막 분류층에서 일부 뉴런을 꺼서 과적합을 줄입니다.
- `PATIENCE=5`: validation F1이 5 epoch 동안 좋아지지 않으면 학습을 멈춥니다.

In [ ]:
def find_data_dir(root):
    # NORMAL, COVID19, PNEUMONIA, TURBERCULOSIS 폴더를 모두 가진 위치를 찾습니다.
    required = {"NORMAL", "COVID19", "PNEUMONIA", "TURBERCULOSIS"}
    root = Path(root)

    for folder in [root, *root.rglob("*")]:
        if folder.is_dir():
            child_names = {p.name for p in folder.iterdir() if p.is_dir()}
            if required.issubset(child_names):
                return folder

    raise RuntimeError("데이터 폴더를 찾지 못했습니다. zip 파일 내부 구조를 확인하세요.")


DATA_DIR = find_data_dir(EXTRACT_DIR)
print("DATA_DIR:", DATA_DIR)

# 이미지 크기입니다. 224는 CNN 이미지 분류에서 많이 쓰는 기본 크기입니다.
# 빠른 테스트만 할 때는 128로 낮추면 학습 속도가 빨라집니다.
IMAGE_SIZE = 224

# 한 번에 학습하는 이미지 개수입니다.
# Colab T4 GPU에서는 32가 안정적이고, GPU RAM이 남으면 64도 시도할 수 있습니다.
BATCH_SIZE = 32

# 전체 train set을 최대 몇 번 반복해서 학습할지 정합니다.
# 최종 학습은 20으로 두고, 테스트 실행은 2~5 정도로 줄이면 됩니다.
EPOCHS = 20

# learning rate는 모델이 한 번 업데이트될 때 얼마나 크게 움직일지 정합니다.
# 1e-4는 AdamW에서 비교적 안정적인 값입니다.
LEARNING_RATE = 1e-4

# weight decay는 모델 가중치가 너무 커지는 것을 막아 과적합을 줄입니다.
WEIGHT_DECAY = 1e-4

# dropout은 분류층의 일부 뉴런을 임시로 꺼서 모델이 데이터를 외우는 것을 줄입니다.
DROPOUT = 0.3

# validation F1이 이 횟수만큼 연속으로 좋아지지 않으면 학습을 조기 종료합니다.
PATIENCE = 5

# 같은 방식으로 데이터를 나누고 학습을 시작하기 위한 난수 고정값입니다.
RANDOM_SEED = 42

NORMAL_LABEL = 0
ABNORMAL_LABEL = 1
LABEL_NAMES = ["NORMAL", "ABNORMAL"]

## 4. 데이터셋과 DataLoader 만들기

`NORMAL` 폴더는 정상, `COVID19`, `PNEUMONIA`, `TURBERCULOSIS` 폴더는 비정상으로 묶습니다. 데이터는 train/validation/test로 나누고, 정상/비정상 비율이 각 set에 비슷하게 들어가도록 stratified split을 사용합니다.

학습 데이터에는 augmentation을 적용합니다. augmentation은 이미지를 약하게 회전하거나 좌우 반전해서, 모델이 특정 이미지 모양을 외우지 않고 더 일반적인 특징을 배우게 하는 방법입니다.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class BinaryXrayDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff")

        for class_name in ["NORMAL", "COVID19", "PNEUMONIA", "TURBERCULOSIS"]:
            class_dir = self.root_dir / class_name
            label = NORMAL_LABEL if class_name == "NORMAL" else ABNORMAL_LABEL

            for file_name in os.listdir(class_dir):
                if file_name.lower().endswith(valid_exts):
                    self.samples.append((class_dir / file_name, label))

        if not self.samples:
            raise RuntimeError("이미지를 찾지 못했습니다.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        image = Image.open(file_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label


class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label = self.subset[idx]
        return self.transform(image), label


def stratified_split(dataset, train_ratio=0.70, val_ratio=0.15, seed=42):
    rng = random.Random(seed)
    label_to_indices = {NORMAL_LABEL: [], ABNORMAL_LABEL: []}

    for idx, (_, label) in enumerate(dataset.samples):
        label_to_indices[label].append(idx)

    train_indices, val_indices, test_indices = [], [], []

    for indices in label_to_indices.values():
        rng.shuffle(indices)
        total = len(indices)
        train_end = int(total * train_ratio)
        val_end = train_end + int(total * val_ratio)

        train_indices.extend(indices[:train_end])
        val_indices.extend(indices[train_end:val_end])
        test_indices.extend(indices[val_end:])

    rng.shuffle(train_indices)
    rng.shuffle(val_indices)
    rng.shuffle(test_indices)
    return train_indices, val_indices, test_indices


set_seed(RANDOM_SEED)

train_transform = transforms.Compose([
    # 모든 이미지를 같은 크기로 맞춰 CNN 입력 크기를 통일합니다.
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    # X-ray 촬영 각도 차이를 흉내 내기 위해 이미지를 최대 10도 정도만 약하게 회전합니다.
    transforms.RandomRotation(10),

    # 좌우 반전을 랜덤하게 적용해 위치 변화에 더 강한 모델을 만듭니다.
    transforms.RandomHorizontalFlip(p=0.5),

    # 병원/기기마다 밝기와 대비가 다를 수 있으므로 아주 약하게 조정합니다.
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),

    # ImageNet 평균/표준편차로 정규화합니다. CNN 학습을 더 안정적으로 만듭니다.
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    # validation/test에는 augmentation을 넣지 않습니다. 평가 데이터는 원래 모습으로 평가해야 합니다.
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

base_dataset = BinaryXrayDataset(DATA_DIR)
counts = {NORMAL_LABEL: 0, ABNORMAL_LABEL: 0}
for _, label in base_dataset.samples:
    counts[label] += 1

train_idx, val_idx, test_idx = stratified_split(base_dataset, seed=RANDOM_SEED)

train_dataset = TransformSubset(Subset(base_dataset, train_idx), train_transform)
val_dataset = TransformSubset(Subset(base_dataset, val_idx), eval_transform)
test_dataset = TransformSubset(Subset(base_dataset, test_idx), eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Label counts:", {LABEL_NAMES[k]: v for k, v in counts.items()})
print(f"Train/Val/Test: {len(train_dataset)} / {len(val_dataset)} / {len(test_dataset)}")

## 5. CNN 모델 만들기

이 모델은 ResNet, EfficientNet 같은 사전학습 모델이 아니라 직접 구성한 custom CNN입니다. `Conv2d -> BatchNorm -> ReLU -> MaxPool` 블록을 반복해서 이미지 특징을 추출하고, 마지막 `Linear` 층에서 정상/비정상 2개 클래스로 분류합니다.

출력값은 확률이 아니라 2개 클래스에 대한 점수(logit)입니다. `CrossEntropyLoss`가 이 점수를 이용해 loss를 계산합니다.

In [ ]:
class BinaryChestCNN(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()

        # features 부분은 이미지에서 선, 음영, 폐 영역 패턴 같은 특징을 추출합니다.
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        # classifier 부분은 추출된 특징을 정상/비정상 2개 class 점수로 바꿉니다.
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(256, 2),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = BinaryChestCNN(dropout=DROPOUT).to(DEVICE)

# 정상 데이터가 비정상 데이터보다 적기 때문에, loss에서 정상 클래스의 가중치를 더 크게 줍니다.
# 이렇게 하면 모델이 데이터가 많은 비정상 쪽으로만 치우치는 것을 줄일 수 있습니다.
total = counts[NORMAL_LABEL] + counts[ABNORMAL_LABEL]
class_weights = torch.tensor([
    total / (2 * counts[NORMAL_LABEL]),
    total / (2 * counts[ABNORMAL_LABEL]),
], dtype=torch.float32, device=DEVICE)

# CrossEntropyLoss는 이진/다중 분류에서 많이 쓰는 loss입니다.
criterion = nn.CrossEntropyLoss(weight=class_weights)

# AdamW는 Adam optimizer에 weight decay를 더 안정적으로 적용한 방식입니다.
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# validation loss가 잘 줄어들지 않으면 learning rate를 절반으로 낮춰 더 섬세하게 학습합니다.
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

print("Class weights:", class_weights.detach().cpu().tolist())

## 6. 학습 함수와 평가 함수

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def evaluate(model, loader):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    tp = tn = fp = fn = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)
            preds = outputs.argmax(dim=1)

            running_loss += loss.item() * images.size(0)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            tp += ((preds == ABNORMAL_LABEL) & (labels == ABNORMAL_LABEL)).sum().item()
            tn += ((preds == NORMAL_LABEL) & (labels == NORMAL_LABEL)).sum().item()
            fp += ((preds == ABNORMAL_LABEL) & (labels == NORMAL_LABEL)).sum().item()
            fn += ((preds == NORMAL_LABEL) & (labels == ABNORMAL_LABEL)).sum().item()

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "loss": running_loss / total,
        "accuracy": correct / total,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

## 7. 학습 실행

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": [], "val_recall": []}
best_val_f1 = 0.0
best_model_state = copy.deepcopy(model.state_dict())
epochs_without_improvement = 0
start_time = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_metrics = evaluate(model, val_loader)
    scheduler.step(val_metrics["loss"])

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_metrics["loss"])
    history["val_acc"].append(val_metrics["accuracy"])
    history["val_f1"].append(val_metrics["f1"])
    history["val_recall"].append(val_metrics["recall"])

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} Train Acc: {train_acc:.4f} "
        f"Val Loss: {val_metrics['loss']:.4f} Val Acc: {val_metrics['accuracy']:.4f} "
        f"Val F1: {val_metrics['f1']:.4f} Val Recall: {val_metrics['recall']:.4f} "
        f"Time: {time.time() - epoch_start:.1f}s"
    )

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping")
        break

print("Total training time:", round(time.time() - start_time, 1), "seconds")

## 8. 테스트 평가와 저장

In [ ]:
model.load_state_dict(best_model_state)
test_metrics = evaluate(model, test_loader)

print("Test Loss:", round(test_metrics["loss"], 4))
print("Test Accuracy:", round(test_metrics["accuracy"], 4))
print("Test Precision:", round(test_metrics["precision"], 4))
print("Test Recall:", round(test_metrics["recall"], 4))
print("Test F1:", round(test_metrics["f1"], 4))
print(f"Confusion Matrix: TN={test_metrics['tn']}, FP={test_metrics['fp']}, FN={test_metrics['fn']}, TP={test_metrics['tp']}")

MODEL_PATH = "/content/drive/MyDrive/chest_binary_cnn_best.pth"
torch.save({
    "model_state_dict": model.state_dict(),
    "image_size": IMAGE_SIZE,
    "label_names": LABEL_NAMES,
    "test_metrics": test_metrics,
}, MODEL_PATH)

print("Saved model:", MODEL_PATH)

## 9. 학습 그래프 확인

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(epochs, history["train_loss"], label="Train")
plt.plot(epochs, history["val_loss"], label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(epochs, history["train_acc"], label="Train")
plt.plot(epochs, history["val_acc"], label="Validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(epochs, history["val_f1"], label="Validation F1")
plt.plot(epochs, history["val_recall"], label="Validation Recall")
plt.title("Validation Metrics")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()